In [ ]:
import h5py
import numpy as np
import os
import matplotlib.pyplot as plt
from scipy.ndimage import label
from skimage import morphology
from skimage import measure
from scipy import ndimage as ndi
from skimage.segmentation import find_boundaries
from scipy.stats import gaussian_kde
from matplotlib.patches import FancyArrowPatch
from scipy.optimize import curve_fit
from scipy.ndimage import map_coordinates
import seaborn as sns
from scipy.ndimage import maximum_filter, label
from scipy.stats import skewnorm
from scipy.stats import lognorm
import fifpy
from scipy.ndimage import gaussian_filter
from scipy.ndimage import label, gaussian_filter
from scipy.stats import lognorm, gaussian_kde
import csv
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, as_completed
from time import perf_counter
from matplotlib.patches import Circle, FancyArrowPatch
from skimage import morphology, measure
try:
    from threadpoolctl import threadpool_limits
except ImportError:
    threadpool_limits = None

plt.rcParams.update({
    'font.size': 20,
    'axes.titlesize': 20,
    'axes.labelsize': 20,
    'xtick.labelsize': 20,
    'ytick.labelsize': 20,
    'legend.fontsize': 20,
    'figure.titlesize': 25
})

# shared code and paths (see README.md and shared/)
import sys
from pathlib import Path
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "shared" / "paths.py").exists())
sys.path.insert(0, str(REPO))
from shared.detection import detect_current_sheets
from shared.fields import partial_x_fft, partial_y_fft
from shared.filtering import iterative_filtering1D, iterative_filtering3D
from shared.thickness import gaussian
from shared.paths import DATA_ROOT, NAS_ROOT, results_dir, cache_dir as shared_cache_dir
RESULTS = results_dir("cs_identification")
CACHE = shared_cache_dir("cs_identification")

In [ ]:
#Defining all the required functions

#Fourier Transform Functions


#Defining constants
dx = 0.0625
dy = 0.0625





In [ ]:
# =====================================================================
# Test the per-sheet visual for ONE sheet (display only, no saving).
# Runs your detection + M1/M2/M3 on one snapshot, picks one sheet, and
# shows the same card as the pooling script via plt.show().
#
# Needs in your session: iterative_filtering3D, partial_x_fft,
#                         partial_y_fft, detect_current_sheets, dx, dy
# =====================================================================

import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch
import h5py
from skimage import morphology, measure
from scipy import ndimage as ndi
from scipy.ndimage import gaussian_filter
from scipy.optimize import curve_fit

# ---- settings ----
beta      = 1
time      = 200
SHEET_K   = 8        # None = largest sheet; or set an integer label to pick a specific one
cache_dir = str(CACHE / "filtered_cache_Jz")
PX_PER_DI = 16.0
PAD       = 20

FILTER_PARAMS_BETA = {0.0625: (3, 2.0), 0.25: (3, 2.3), 1: (3, 2.5), 4: (3, 2.7), 16: (3, 3.0)}
max_len, subpix_step, margin_frac, min_half, smooth_sigma = 60, 0.5, 0.5, 6.0, 3

base_path  = f"{DATA_ROOT}/beta_p_{beta}/beta_e_1/"
n_iter, Xi = FILTER_PARAMS_BETA[beta]


# ---- helpers (identical to the pooling script) ----


def _walk_m2(mask, x0, y0, dx_, dy_, step=1, max_steps=1000):
    nx, ny = mask.shape
    xf, yf, s = float(x0), float(y0), 0
    while s < max_steps:
        xn, yn = xf + dx_ * step, yf + dy_ * step
        xi, yi = int(round(xn)), int(round(yn))
        if xi < 0 or yi < 0 or xi >= nx or yi >= ny:
            break
        if not mask[xi, yi]:
            break
        xf, yf, s = xn, yn, s + 1
    return np.hypot(xf - x0, yf - y0), (xf, yf)


def _m3_profile_and_fit(J2, mask, xm, ym, dx_, dy_):
    if dx_ == 0 and dy_ == 0:
        return None, None, None, np.nan, np.nan
    nx, ny = mask.shape

    def edge(sx, sy):
        d = 0.0
        while d < max_len:
            dn = d + subpix_step
            xi, yi = int(round(xm + sx * dn)) % nx, int(round(ym + sy * dn)) % ny
            if not mask[xi, yi]:
                break
            d = dn
        return d
    half_f = min(max_len, max(min_half, edge(dx_, dy_) * (1 + margin_frac)))
    half_b = min(max_len, max(min_half, edge(-dx_, -dy_) * (1 + margin_frac)))
    xs = np.arange(-half_b, half_f + subpix_step, subpix_step)
    vals = ndi.map_coordinates(J2, np.vstack([xm + xs * dx_, ym + xs * dy_]),
                               order=1, mode="grid-wrap")
    if len(vals) < 4:
        return xs, vals, None, np.nan, np.nan
    B0, A0 = float(vals.min()), float(vals.max() - vals.min())
    if A0 <= 0:
        return xs, vals, None, np.nan, np.nan
    mu0 = float(xs[np.argmax(vals)])
    above = np.where(vals >= B0 + 0.5 * A0)[0]
    sigma0 = (max(1.0, (xs[above[-1]] - xs[above[0]]) / 2.3548)
              if above.size >= 2 else max(2.0, (half_f + half_b) / 6.0))
    span = float(xs.max() - xs.min())
    sigma0 = min(sigma0, 0.99 * span)
    bounds = ([0.0, xs.min() - 1, 1e-3, -np.inf], [np.inf, xs.max() + 1, span, np.inf])
    try:
        popt, _ = curve_fit(gaussian, xs, vals, p0=[A0, mu0, sigma0, B0],
                            bounds=bounds, maxfev=10000)
        return xs, vals, popt, 2 * np.sqrt(2 * np.log(2)) * abs(popt[2]), popt[1]
    except (RuntimeError, ValueError):
        return xs, vals, None, np.nan, np.nan


# ---- load B + filter (cached) ----
with h5py.File(base_path + f"Bx_ApJ_t{time}.h5", "r") as fBx, \
     h5py.File(base_path + f"By_ApJ_t{time}.h5", "r") as fBy, \
     h5py.File(base_path + f"Bz_ApJ_t{time}.h5", "r") as fBz:
    data_Bx = fBx["DS1"][:].T
    data_By = fBy["DS1"][:].T
    data_Bz = fBz["DS1"][:].T

cache_file = f"{cache_dir}/filtered_beta_{beta}_t{time}_iter{n_iter}_Xi{Xi}.npz"
if os.path.exists(cache_file):
    d = np.load(cache_file)
    data_Bx_f, data_By_f, data_Bz_f = d["Bx"], d["By"], d["Bz"]
else:
    data_Bx_f, data_By_f, data_Bz_f = iterative_filtering3D(data_Bx, data_By, data_Bz, n_iter, Xi=Xi)
    np.savez_compressed(cache_file, Bx=data_Bx_f, By=data_By_f, Bz=data_Bz_f)

# ---- current, detect, clean, label ----
Jz = partial_x_fft(data_By_f, dy) - partial_y_fft(data_Bx_f, dx)
J2 = np.abs(Jz)

sheets = detect_current_sheets(J2)
mask = np.zeros_like(J2)
for sheet in sheets:
    for (i, j) in sheet:
        mask[i, j] = 1
struct = morphology.disk(1)
mask = morphology.binary_opening(mask, struct)
mask = morphology.binary_closing(mask, struct)
cleaned_mask = morphology.remove_small_objects(mask.astype(bool), min_size=500).astype(int)

labeled = measure.label(cleaned_mask, connectivity=2)
nreg = int(labeled.max())
if nreg == 0:
    raise SystemExit("no sheets found in this snapshot")

# pick the sheet: largest by default
if SHEET_K is None:
    sizes = np.bincount(labeled.ravel())
    sizes[0] = 0                       # ignore background
    SHEET_K = int(np.argmax(sizes))
print(f"{nreg} sheets found; showing sheet {SHEET_K} "
      f"(size {int((labeled == SHEET_K).sum())} px)")

J2_smooth = gaussian_filter(J2, sigma=smooth_sigma)
gx, gy = np.gradient(J2_smooth)

# ---- measure M1/M2/M3 for this one sheet ----
region = (labeled == SHEET_K)
xs_r, ys_r = np.where(region)
p = int(np.argmax(J2[region]))
xm, ym = int(xs_r[p]), int(ys_r[p])

edt = ndi.distance_transform_edt(region)
r1 = float(edt.max()); m1 = 2 * r1
epx, epy = np.unravel_index(int(np.argmax(edt)), edt.shape)

gvec = np.array([gx[xm, ym], gy[xm, ym]])
if np.allclose(gvec, 0):
    dx_, dy_ = 0.0, 0.0
else:
    v = -gvec / np.linalg.norm(gvec); dx_, dy_ = float(v[0]), float(v[1])

df, fwd = _walk_m2(cleaned_mask, xm, ym, dx_, dy_)
db, bwd = _walk_m2(cleaned_mask, xm, ym, -dx_, -dy_)
m2 = df + db

xs_p, vals_p, popt, fwhm, mu = _m3_profile_and_fit(J2, cleaned_mask, xm, ym, dx_, dy_)

print(f"  M1 = {m1/PX_PER_DI:.3f} d_i   M2 = {m2/PX_PER_DI:.3f} d_i   "
      f"M3 = {fwhm/PX_PER_DI:.3f} d_i")

# ---- render the card (display only, no saving) ----
S = 1.0 / PX_PER_DI                       # pixels -> d_p  (full box: Nx/PX_PER_DI = 256)
Nx, Ny = J2.shape
x0, x1 = max(0, xs_r.min() - PAD), min(Nx, xs_r.max() + PAD + 1)
y0, y1 = max(0, ys_r.min() - PAD), min(Ny, ys_r.max() + PAD + 1)
crop = J2[x0:x1, y0:y1]
regcrop = (labeled[x0:x1, y0:y1] == SHEET_K)
ext = [x0 * S, x1 * S, y0 * S, y1 * S]    # extent in d_p

fig, (axL, axR) = plt.subplots(1, 2, figsize=(11, 5))
axL.imshow(crop.T, origin="lower", extent=ext, cmap="inferno",
           vmax=float(np.percentile(crop, 99.5)), aspect="equal")
axL.contour(regcrop.T, levels=[0.5], colors="white", linewidths=1.0, extent=ext)
axL.plot(xm * S, ym * S, "x", color="yellow", ms=11, mew=2, label="max", zorder=6)
axL.add_patch(Circle((epx * S, epy * S), r1 * S, fill=False, ec="cyan", lw=1.6,
                     label=f"M1 = {m1/PX_PER_DI:.2f}"))
(xf, yf), (xb, yb) = fwd, bwd
axL.plot([xb * S, xf * S], [yb * S, yf * S], "-", color="lime", lw=2.2,
         label=f"M2 = {m2/PX_PER_DI:.2f}", zorder=5)
if np.isfinite(fwhm) and (dx_ or dy_):
    cx, cy = (xm + mu * dx_) * S, (ym + mu * dy_) * S
    ex, ey = (fwhm / 2) * dx_ * S, (fwhm / 2) * dy_ * S
    axL.plot([cx - ex, cx + ex], [cy - ey, cy + ey], "-", color="magenta",
             lw=2.2, label=f"M3 = {fwhm/PX_PER_DI:.2f}", zorder=5)
if dx_ or dy_:
    axL.add_patch(FancyArrowPatch((xm * S, ym * S), ((xm + 8 * dx_) * S, (ym + 8 * dy_) * S),
                  arrowstyle="->", color="white", mutation_scale=12, lw=1, zorder=6))
axL.set_xlim(x0 * S, x1 * S); axL.set_ylim(y0 * S, y1 * S)
axL.set_xlabel(r"$x/d_p$"); axL.set_ylabel(r"$y/d_p$")
axL.set_title(f"sheet {SHEET_K}")
axL.legend(fontsize=12, loc="upper right", framealpha=0.6)

if xs_p is not None and len(xs_p) > 0:
    axR.plot(xs_p * S, vals_p, "o", ms=3, color="0.3", label=r"$|J_\parallel|$ profile")
    if popt is not None:
        A, muf, sig, B = popt
        xd = np.linspace(xs_p.min(), xs_p.max(), 400)
        axR.plot(xd * S, gaussian(xd, A, muf, sig, B), "-", color="magenta", label="Gaussian fit")
        axR.axhline(B + A / 2, color="0.35", ls="--", lw=1.5, label="half-max")
    L = max(abs(xs_p.min()), abs(xs_p.max())) * S     # symmetric x-range about the peak
    axR.set_xlim(-L, L)
    axR.set_xlabel(r"distance along descent ($d_i$)"); axR.set_ylabel(r"$|J_\parallel|$")
    axR.legend(fontsize=15)
else:
    axR.text(0.5, 0.5, "no profile (flat gradient)", ha="center", va="center",
             transform=axR.transAxes, color="0.5")

fig.suptitle(rf"$\beta_p = {beta}$,  $\beta_e = 1$,  $t = {time}$,  sheet {SHEET_K}   |   "
             rf"M1 = {m1/PX_PER_DI:.2f}  M2 = {m2/PX_PER_DI:.2f}  "
             rf"M3 = {fwhm/PX_PER_DI:.2f}  $d_i$", fontsize=25)
fig.tight_layout()
plt.savefig(RESULTS / "SingleSheet.png", dpi=300, bbox_inches='tight')
plt.show()